In [ ]:
PRESET      = "fast"        # smoke | minimal | fast | full
RUN_DIR     = "/kaggle/working/run"
RESUME_FROM = None
STAGES      = "setup,gen0,pool,chains,evaluate" 
CHAINS      = None     
MODEL       = None   

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "diffusers>=0.30" 2>&1 | tail -1
import torch; print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
import os; os.makedirs('/kaggle/working/recursive-density-bias/rdb', exist_ok=True); os.chdir('/kaggle/working/recursive-density-bias')

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/rdb/__init__.py
__version__ = "1.0.0"

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/rdb/config.py
import copy

CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

DEFAULT_SPLIT = {
    "airplane": 5000, "automobile": 4900, "frog": 4950, "horse": 4850, "ship": 4800,  
    "deer": 1500, "bird": 800, "dog": 400, "cat": 150, "truck": 50,                   
}
HEAD_CLASSES = ["airplane", "automobile", "frog", "horse", "ship"]
TAIL_CLASSES = ["deer", "bird", "dog", "cat", "truck"]

MODEL_CFGS = {
    "base": dict(block_out_channels=[64, 128, 256, 256], layers_per_block=2,
                 attn=[False, True, False, False], dropout=0.1, head_dim=32, groups=32),

    "small": dict(block_out_channels=[64, 128, 128, 256], layers_per_block=2,
                  attn=[False, True, False, False], dropout=0.1, head_dim=32, groups=32),

    "large": dict(block_out_channels=[96, 192, 288, 288], layers_per_block=2,
                  attn=[False, True, False, False], dropout=0.1, head_dim=32, groups=32),

    "tiny": dict(block_out_channels=[32, 64, 64, 64], layers_per_block=1,
                 attn=[False, True, False, False], dropout=0.0, head_dim=32, groups=16),
}

CHAINS = {
    "baseline": dict(strategy="random", pool_mult=1.0),

    "kdr": dict(strategy="kdr", pool_mult=2.0, k=50, gamma=1.0, alpha=1.0, clip_q=0.005),
    
    "oracle": dict(strategy="oracle", pool_mult=2.0),
    
    "realmix": dict(strategy="realmix", pool_mult=1.0, real_frac=0.5),
    
    "flatten": dict(strategy="flatten", pool_mult=2.0, k=10, gamma=0.5, clip_q=0.01),
}

BASE = dict(
    seed=0,
    split=DEFAULT_SPLIT,
    head=HEAD_CLASSES,
    tail=TAIL_CLASSES,
    model="base",
    schedule="linear",       
    T=1000,

    gen0_steps=40000, gen0_batch=128, gen0_lr=2e-4,

    ft_steps=15000, ft_batch=128, ft_lr=1e-4,
    warmup=1000, ema=0.9995, grad_clip=1.0,

    ddim_steps=100, sample_batch=1000, noise_scale=1.0,

    gens=2,
    chains=CHAINS,
    run_chains=["baseline", "kdr", "oracle", "realmix"],

    clf_epochs=30, clf_batch=256, conf_thresh=0.5,

    feat_backbone="auto",     
    knn_k=5,                 
    data_frac=1.0,          
)

PRESETS = {
    "full": dict(),
    "fast": dict(gen0_steps=20000, ft_steps=8000, ddim_steps=50, clf_epochs=20,
                 run_chains=["baseline", "kdr", "oracle", "realmix"]),
    "minimal": dict(gen0_steps=20000, ft_steps=8000, ddim_steps=50, clf_epochs=15,
                    run_chains=["baseline", "kdr"]),
    "smoke": dict(gen0_steps=200, ft_steps=100, warmup=20, ddim_steps=8, clf_epochs=2,
                  model="tiny", data_frac=0.1, gen0_batch=32, ft_batch=32, sample_batch=256,
                  run_chains=["baseline", "kdr", "oracle", "realmix", "flatten"],
                  chains={**CHAINS, "kdr": dict(CHAINS["kdr"], k=10)}),
}


def make_config(preset="full", **overrides):
    cfg = copy.deepcopy(BASE)
    cfg.update(copy.deepcopy(PRESETS[preset]))
    cfg["preset"] = preset
    for k, v in overrides.items():
        if v is not None:
            cfg[k] = v
    return cfg

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/rdb/common.py

import json
import os
import random
import sys
import time

import numpy as np
import torch


def jload(path):
    with open(path) as f:
        return json.load(f)


def jsave(obj, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)
    os.replace(tmp, path)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(type(o))


def npsave(path, arr):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = path[:-4] + ".tmp.npy"
    np.save(tmp, arr)
    os.replace(tmp, path)


def load_cfg(run):
    return jload(os.path.join(run, "config.json"))


def save_cfg(run, cfg):
    jsave(cfg, os.path.join(run, "config.json"))


def get_device():
    if torch.cuda.is_available():
        lr = int(os.environ.get("LOCAL_RANK", 0))
        return torch.device(f"cuda:{lr}")
    return torch.device("cpu")


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def u8_to_float(x_u8_nhwc_torch):
    return x_u8_nhwc_torch.permute(0, 3, 1, 2).float().div(127.5).sub(1.0)


def float_to_u8(x):
    return ((x.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8).permute(0, 2, 3, 1).cpu().numpy()


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)
    sys.stdout.flush()


def save_grid(imgs_u8, path, nrow=10, pad=2):
    from PIL import Image
    n = len(imgs_u8)
    if n == 0:
        return
    ncol = min(nrow, n)
    nr = (n + ncol - 1) // ncol
    h, w = imgs_u8.shape[1:3]
    canvas = np.full((nr * (h + pad) + pad, ncol * (w + pad) + pad, 3), 255, np.uint8)
    for i, im in enumerate(imgs_u8):
        r, c = divmod(i, ncol)
        canvas[pad + r * (h + pad): pad + r * (h + pad) + h, pad + c * (w + pad): pad + c * (w + pad) + w] = im
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    Image.fromarray(canvas).resize((canvas.shape[1] * 2, canvas.shape[0] * 2), Image.NEAREST).save(path)


def fmt_secs(s):
    s = int(s)
    return f"{s // 3600}h{(s % 3600) // 60:02d}m" if s >= 3600 else f"{s // 60}m{s % 60:02d}s"

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/rdb/data.py
import glob
import os
import pickle
import tarfile

import numpy as np

from .common import jsave, log, npsave, save_grid
from .config import CIFAR10_CLASSES


def _load_batches(folder):
    def rd(f):
        with open(f, "rb") as fh:
            d = pickle.load(fh, encoding="latin1")
        x = np.asarray(d["data"], np.uint8).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        return x, np.asarray(d["labels"], np.int64)
    xs, ys = zip(*[rd(os.path.join(folder, f"data_batch_{i}")) for i in range(1, 6)])
    xt, yt = rd(os.path.join(folder, "test_batch"))
    return np.concatenate(xs), np.concatenate(ys), xt, yt


def _find_cifar_folder(roots):
    for root in roots:
        if not root or not os.path.exists(root):
            continue
        hits = glob.glob(os.path.join(root, "**", "cifar-10-batches-py"), recursive=True)
        if hits:
            return hits[0]
        tars = glob.glob(os.path.join(root, "**", "cifar-10-python.tar.gz"), recursive=True)
        if tars:
            dest = os.path.join("/tmp", "cifar_extract")
            os.makedirs(dest, exist_ok=True)
            with tarfile.open(tars[0]) as t:
                t.extractall(dest)
            return os.path.join(dest, "cifar-10-batches-py")
    return None


def fake_cifar(n_train=5000, n_test=1000, seed=0):
    rng = np.random.default_rng(seed)
    colors = rng.integers(40, 215, size=(10, 3))
    yy, xx = np.mgrid[0:32, 0:32]

    def make(n):
        X, Y = [], []
        for c in range(10):
            ang = np.pi * c / 10
            freq = 0.25 + 0.05 * (c % 3)
            for _ in range(n):
                ph = rng.uniform(0, 2 * np.pi)
                wave = np.sin(freq * (np.cos(ang) * xx + np.sin(ang) * yy) + ph)[..., None]
                img = colors[c] + 35 * wave + rng.normal(0, 12, (32, 32, 3))
                X.append(np.clip(img, 0, 255).astype(np.uint8))
                Y.append(c)
        return np.stack(X), np.asarray(Y, np.int64)

    xtr, ytr = make(n_train)
    xte, yte = make(n_test)
    return xtr, ytr, xte, yte


def load_cifar10(source="auto", root="./cifar_data"):
    if source == "fake":
        log("Using FAKE CIFAR-like data (smoke test only)")
        return fake_cifar(n_train=600, n_test=100)
    folder = _find_cifar_folder([root, "/kaggle/input"])
    if folder is None:
        import torchvision
        log("Downloading CIFAR-10 with torchvision ...")
        torchvision.datasets.CIFAR10(root=root, train=True, download=True)
        folder = _find_cifar_folder([root])
    log("CIFAR-10 from", folder)
    return _load_batches(folder)


def build_split(x, y, split, seed=0, frac=1.0):
    rng = np.random.default_rng(seed)
    idx_all = []
    for c, name in enumerate(CIFAR10_CLASSES):
        want = max(1, int(round(split[name] * frac)))
        cand = np.where(y == c)[0]
        if want > len(cand):
            raise ValueError(f"class {name}: want {want}, only {len(cand)} available")
        idx_all.append(np.sort(rng.choice(cand, want, replace=False)))
    idx = np.concatenate(idx_all)
    perm = rng.permutation(len(idx))
    idx = idx[perm]
    return x[idx], y[idx], idx


def prepare_data(run, cfg, source="auto", cifar_root="./cifar_data"):
    d = os.path.join(run, "data")
    os.makedirs(d, exist_ok=True)
    if os.path.exists(os.path.join(d, "real_train.npy")):
        log("data already prepared")
        return
    xtr, ytr, xte, yte = load_cifar10(source, cifar_root)
    xs, ys, idx = build_split(xtr, ytr, cfg["split"], cfg["seed"], cfg["data_frac"])
    npsave(os.path.join(d, "real_train.npy"), xs)
    npsave(os.path.join(d, "real_labels.npy"), ys)
    npsave(os.path.join(d, "split_indices.npy"), idx)

    npsave(os.path.join(d, "cifar_train.npy"), xtr)
    npsave(os.path.join(d, "cifar_train_labels.npy"), ytr)
    npsave(os.path.join(d, "cifar_test.npy"), xte)
    npsave(os.path.join(d, "cifar_test_labels.npy"), yte)

    counts = np.bincount(ys, minlength=10)
    p0 = counts / counts.sum()
    table = []
    for c, name in enumerate(CIFAR10_CLASSES):
        group = "head" if name in cfg["head"] else "tail"
        table.append(dict(class_id=c, cls=name, group=group, count=int(counts[c]), proportion=float(p0[c])))
    info = dict(
        source=source, seed=cfg["seed"], total=int(counts.sum()),
        imbalance_ratio=float(counts.max() / counts.min()),
        head_share=float(sum(t["proportion"] for t in table if t["group"] == "head")),
        tail_share=float(sum(t["proportion"] for t in table if t["group"] == "tail")),
        classes=table,
        note="Indices refer to the official CIFAR-10 training set order (data_batch_1..5).",
    )
    jsave(info, os.path.join(d, "split.json"))
    with open(os.path.join(d, "split_table.csv"), "w") as f:
        f.write("class_id,class,group,count,proportion\n")
        for t in table:
            f.write(f"{t['class_id']},{t['cls']},{t['group']},{t['count']},{t['proportion']:.5f}\n")

    ex = []
    for c in range(10):
        ex.extend(xs[ys == c][:8])
    save_grid(np.stack(ex), os.path.join(run, "results", "grids", "real_split_examples.png"), nrow=8)
    log(f"split: {counts.sum()} images, imbalance {info['imbalance_ratio']:.0f}:1, "
        f"tail share {info['tail_share']:.3f}")
    for t in table:
        log(f"   {t['cls']:<11}{t['group']:<6}{t['count']:>6}  ({t['proportion']:.4f})")

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/rdb/diffusion.py

import copy
import math
import os
import time

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F

from .common import float_to_u8, fmt_secs, get_device, log, save_grid, u8_to_float


def build_unet(mcfg):
    from diffusers import UNet2DModel
    attn = mcfg["attn"]
    down = tuple("AttnDownBlock2D" if a else "DownBlock2D" for a in attn)
    up = tuple("AttnUpBlock2D" if a else "UpBlock2D" for a in reversed(attn))
    return UNet2DModel(
        sample_size=32, in_channels=3, out_channels=3,
        layers_per_block=mcfg["layers_per_block"],
        block_out_channels=tuple(mcfg["block_out_channels"]),
        down_block_types=down, up_block_types=up,
        dropout=mcfg.get("dropout", 0.0),
        attention_head_dim=mcfg.get("head_dim", 32),
        norm_num_groups=mcfg.get("groups", 32),
    )


def load_model(ckpt_path, device):
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model = build_unet(sd["mcfg"])
    model.load_state_dict(sd["ema"])
    return model.to(device).eval(), sd



def alphas_cumprod(T=1000, kind="linear"):
    if kind == "linear":
        betas = torch.linspace(1e-4, 0.02, T, dtype=torch.float64)
    elif kind == "cosine":
        s = 0.008
        t = torch.linspace(0, T, T + 1, dtype=torch.float64) / T
        f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
        betas = (1 - f[1:] / f[:-1]).clamp(max=0.999)
    else:
        raise ValueError(kind)
    return torch.cumprod(1 - betas, 0).float()


@torch.no_grad()
def ddim_sample(model, n, steps=100, T=1000, schedule="linear", batch=1000, seed=0,
                noise_scale=1.0, device=None, progress=True):

    device = device or next(model.parameters()).device
    ac = alphas_cumprod(T, schedule).to(device)
    ts = torch.linspace(T - 1, 0, steps).round().long().tolist()
    use_amp = device.type == "cuda"
    gen = torch.Generator(device=device).manual_seed(int(seed))
    out, t0 = [], time.time()
    for b0 in range(0, n, batch):
        m = min(batch, n - b0)
        x = torch.randn(m, 3, 32, 32, generator=gen, device=device) * noise_scale
        for i, t in enumerate(ts):
            tb = torch.full((m,), t, device=device, dtype=torch.long)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                eps = model(x, tb).sample
            eps = eps.float()
            a_t = ac[t]
            a_prev = ac[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=device)
            x0 = ((x - (1 - a_t).sqrt() * eps) / a_t.sqrt()).clamp(-1, 1)
            eps = (x - a_t.sqrt() * x0) / (1 - a_t).sqrt()
            x = a_prev.sqrt() * x0 + (1 - a_prev).sqrt() * eps
        out.append(float_to_u8(x))
        if progress:
            done = b0 + m
            el = time.time() - t0
            log(f"  sampled {done}/{n}  ({el / done * 1000:.1f} ms/img, ETA {fmt_secs(el / done * (n - done))})")
    return np.concatenate(out)



def _dist_setup():
    if int(os.environ.get("WORLD_SIZE", 1)) > 1 and not dist.is_initialized():
        backend = "nccl" if torch.cuda.is_available() else "gloo"
        dist.init_process_group(backend)
        if torch.cuda.is_available():
            torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
    if dist.is_initialized():
        return dist.get_rank(), dist.get_world_size()
    return 0, 1


def train_diffusion(data_path, out_dir, mcfg, steps, batch, lr, warmup=1000, ema_decay=0.9995,
                    grad_clip=1.0, init=None, seed=0, T=1000, schedule="linear",
                    log_every=200, save_every=2000, grid_steps=50, meta=None):
                    
    rank, world = _dist_setup()
    device = get_device()
    final_path = os.path.join(out_dir, "ema.pt")
    last_path = os.path.join(out_dir, "ckpt_last.pt")
    if os.path.exists(final_path):
        if rank == 0:
            log(f"{final_path} exists - skipping training")
        return final_path
    os.makedirs(out_dir, exist_ok=True)

    data = torch.from_numpy(np.load(data_path)).to(device)   
    N = data.shape[0]
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

    model = build_unet(mcfg).to(device)
    if init is not None:
        sd = torch.load(init, map_location="cpu", weights_only=False)
        model.load_state_dict(sd["ema"])
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0, betas=(0.9, 0.999))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    start = 0
    if os.path.exists(last_path):
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"]); ema.load_state_dict(ck["ema"])
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start = ck["step"]
        if rank == 0:
            log(f"resuming from step {start}")
    nparams = sum(p.numel() for p in model.parameters())
    if rank == 0:
        log(f"train: N={N}, steps={steps}, batch={batch}x{world}, lr={lr}, params={nparams / 1e6:.1f}M, "
            f"init={'yes' if init else 'no'}, device={device}")

    net = model
    if world > 1:
        net = torch.nn.parallel.DistributedDataParallel(
            model, device_ids=[device.index] if device.type == "cuda" else None)

    ac = alphas_cumprod(T, schedule).to(device)
    g = torch.Generator(device=device).manual_seed(seed * 7919 + rank * 104729 + start)
    ema_params = [p for p in ema.parameters()]
    model_params = [p for p in model.parameters()]
    t_start, loss_acc, n_acc, history = time.time(), 0.0, 0, []
    net.train()
    for step in range(start, steps):
        for pg in opt.param_groups:
            pg["lr"] = lr * min(1.0, (step + 1) / max(1, warmup))
        idx = torch.randint(0, N, (batch,), device=device, generator=g)
        x0 = u8_to_float(data[idx])
        flip = torch.rand(batch, device=device, generator=g) < 0.5
        x0 = torch.where(flip[:, None, None, None], x0.flip(3), x0)
        t = torch.randint(0, T, (batch,), device=device, generator=g)
        noise = torch.randn(x0.shape, device=device, generator=g)
        a = ac[t].view(-1, 1, 1, 1)
        xt = a.sqrt() * x0 + (1 - a).sqrt() * noise
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            pred = net(xt, t).sample
        loss = F.mse_loss(pred.float(), noise)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(opt)
        scaler.update()
        k = step - start + 1
        d = min(ema_decay, (1 + k) / (10 + k))
        with torch.no_grad():
            torch._foreach_lerp_(ema_params, model_params, 1 - d)
        loss_acc = loss_acc + loss.detach()
        n_acc += 1
        if rank == 0 and ((step + 1) % log_every == 0 or step + 1 == steps):
            el = time.time() - t_start
            ips = (step + 1 - start) / el
            l = float(loss_acc) / max(1, n_acc)
            history.append((step + 1, l))
            log(f"  step {step + 1}/{steps}  loss {l:.4f}  {ips:.2f} it/s  ETA {fmt_secs((steps - step - 1) / ips)}")
            loss_acc, n_acc = 0.0, 0
        if rank == 0 and (step + 1) % save_every == 0 and step + 1 < steps:
            torch.save(dict(model=model.state_dict(), ema=ema.state_dict(), opt=opt.state_dict(),
                            scaler=scaler.state_dict(), step=step + 1), last_path + ".tmp")
            os.replace(last_path + ".tmp", last_path)

    if rank == 0:
        sd = dict(ema=ema.state_dict(), mcfg=mcfg, steps=steps, data=data_path, init=init,
                  T=T, schedule=schedule, meta=meta or {}, train_time_s=time.time() - t_start)
        torch.save(sd, final_path + ".tmp")
        os.replace(final_path + ".tmp", final_path)
        with open(os.path.join(out_dir, "train_log.csv"), "a") as f:
            for s, l in history:
                f.write(f"{s},{l:.6f}\n")
        if os.path.exists(last_path):
            os.remove(last_path)
        imgs = ddim_sample(ema, 64, steps=grid_steps, T=T, schedule=schedule, batch=64, seed=1234,
                           device=device, progress=False)
        save_grid(imgs, os.path.join(out_dir, "samples_grid.png"), nrow=8)
        log(f"saved {final_path}")
    if dist.is_initialized():
        dist.barrier()
        dist.destroy_process_group()
    return final_path

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/rdb/classifier.py

import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

from .common import get_device, jsave, log, seed_all

MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)


def resnet18_cifar():
    m = torchvision.models.resnet18(num_classes=10)
    m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    m.maxpool = nn.Identity()
    return m


def _prep(x_u8_nhwc, device):
    x = x_u8_nhwc.to(device).permute(0, 3, 1, 2).float() / 255.0
    return (x - MEAN.to(device)) / STD.to(device)


def _augment(x, gen):
    """Random crop (pad 4, reflect) + horizontal flip, fully on GPU."""
    B = x.shape[0]
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i = torch.randint(0, 9, (B,), device=x.device, generator=gen)
    j = torch.randint(0, 9, (B,), device=x.device, generator=gen)
    ar = torch.arange(32, device=x.device)
    rows = (i[:, None] + ar[None, :])[:, None, :, None].expand(B, 3, 32, 32)
    cols = (j[:, None] + ar[None, :])[:, None, None, :].expand(B, 3, 32, 32)
    out = xp[torch.arange(B, device=x.device)[:, None, None, None],
             torch.arange(3, device=x.device)[None, :, None, None], rows, cols]
    flip = torch.rand(B, device=x.device, generator=gen) < 0.5
    return torch.where(flip[:, None, None, None], out.flip(3), out)


def train_classifier(run, cfg):
    out = os.path.join(run, "classifier")
    path = os.path.join(out, "resnet18.pt")
    if os.path.exists(path):
        log("classifier exists - skipping")
        return path
    os.makedirs(out, exist_ok=True)
    seed_all(cfg["seed"])
    device = get_device()
    d = os.path.join(run, "data")
    xtr = torch.from_numpy(np.load(os.path.join(d, "cifar_train.npy")))
    ytr = torch.from_numpy(np.load(os.path.join(d, "cifar_train_labels.npy")))
    xte = np.load(os.path.join(d, "cifar_test.npy"))
    yte = np.load(os.path.join(d, "cifar_test_labels.npy"))
    X = _prep(xtr, device)
    Y = ytr.to(device)
    model = resnet18_cifar().to(device).to(memory_format=torch.channels_last)
    E, B = cfg["clf_epochs"], cfg["clf_batch"]
    steps_per_epoch = len(X) // B
    opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=0.1, total_steps=E * steps_per_epoch,
                                                pct_start=0.25)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    gen = torch.Generator(device=device).manual_seed(cfg["seed"])
    t0 = time.time()
    for ep in range(E):
        model.train()
        perm = torch.randperm(len(X), device=device, generator=gen)
        tot, correct, lsum = 0, 0, 0.0
        for s in range(steps_per_epoch):
            idx = perm[s * B:(s + 1) * B]
            xb = _augment(X[idx], gen).contiguous(memory_format=torch.channels_last)
            yb = Y[idx]
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            lsum += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            tot += len(yb)
        log(f"  clf epoch {ep + 1}/{E} loss {lsum / tot:.3f} train-acc {correct / tot:.3f} ({time.time() - t0:.0f}s)")
    torch.save(model.state_dict(), path)
    probs = predict_probs(xte, model=model, device=device)
    pred = probs.argmax(1)
    acc = float((pred == yte).mean())
    conf = np.zeros((10, 10))
    for t, p in zip(yte, pred):
        conf[t, p] += 1
    conf = conf / conf.sum(1, keepdims=True)
    np.save(os.path.join(out, "confusion_test.npy"), conf)
    keep = probs.max(1) >= cfg["conf_thresh"]
    jsave(dict(test_acc=acc, per_class_acc=np.diag(conf).tolist(),
               frac_confident=float(keep.mean()),
               acc_confident=float((pred[keep] == yte[keep]).mean()),
               epochs=E, train_time_s=time.time() - t0), os.path.join(out, "metrics.json"))
    log(f"classifier test accuracy {acc:.4f}")
    return path


_CACHE = {}


def load_classifier(run, device):
    key = (run, str(device))
    if key not in _CACHE:
        m = resnet18_cifar()
        m.load_state_dict(torch.load(os.path.join(run, "classifier", "resnet18.pt"), map_location="cpu"))
        _CACHE[key] = m.to(device).eval()
    return _CACHE[key]


@torch.no_grad()
def predict_probs(imgs_u8, run=None, model=None, device=None, batch=1000, return_feats=False):

    device = device or get_device()
    model = model if model is not None else load_classifier(run, device)
    model.eval()
    P, Fs = [], []
    for i in range(0, len(imgs_u8), batch):
        x = _prep(torch.from_numpy(np.ascontiguousarray(imgs_u8[i:i + batch])), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            h = model.conv1(x); h = model.bn1(h); h = model.relu(h)
            h = model.layer4(model.layer3(model.layer2(model.layer1(h))))
            f = torch.flatten(model.avgpool(h), 1)
            logits = model.fc(f)
        P.append(logits.float().softmax(1).cpu().numpy())
        if return_feats:
            Fs.append(f.float().cpu().numpy())
    P = np.concatenate(P)
    return (P, np.concatenate(Fs)) if return_feats else P

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/rdb/features.py

import os

import numpy as np
import torch
import torch.nn.functional as F

from .common import get_device, load_cfg, log, npsave, save_cfg

IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class FeatureExtractor:
    def __init__(self, backbone, device, run=None):
        self.backbone, self.device, self.run = backbone, device, run
        if backbone == "dinov2_vits14":
            self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", verbose=False)
            self.size = 224
        elif backbone == "timm_dinov2_vits14":
            import timm
            self.model = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True,
                                           num_classes=0, img_size=224)
            self.size = 224
        elif backbone == "classifier":
            from .classifier import load_classifier
            self.model = load_classifier(run, device)
            self.size = 32
        else:
            raise ValueError(backbone)
        self.model = self.model.to(device).eval()

    @torch.no_grad()
    def __call__(self, imgs_u8, batch=256):
        if self.backbone == "classifier":
            from .classifier import predict_probs
            return predict_probs(imgs_u8, model=self.model, device=self.device, return_feats=True)[1]
        out = []
        amp = self.device.type == "cuda"
        for i in range(0, len(imgs_u8), batch):
            x = torch.from_numpy(np.ascontiguousarray(imgs_u8[i:i + batch])).to(self.device)
            x = x.permute(0, 3, 1, 2).float() / 255.0
            x = F.interpolate(x, size=(self.size, self.size), mode="bicubic", align_corners=False).clamp(0, 1)
            x = (x - IMNET_MEAN.to(self.device)) / IMNET_STD.to(self.device)
            with torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=amp):
                f = self.model(x)
            out.append(f.float().cpu().numpy())
        return np.concatenate(out)


_CACHE = {}


def get_extractor(run, device=None):

    device = device or get_device()
    cfg = load_cfg(run)
    bk = cfg.get("feat_backbone_resolved")
    key = (run, str(device))
    if key in _CACHE:
        return _CACHE[key]
    if bk is None:
        order = ["dinov2_vits14", "timm_dinov2_vits14", "classifier"] if cfg["feat_backbone"] == "auto" \
            else [cfg["feat_backbone"]]
        err = None
        for b in order:
            try:
                ext = FeatureExtractor(b, device, run)
                bk = b
                break
            except Exception as e:
                err = e
                log(f"feature backbone {b} unavailable: {type(e).__name__}: {str(e)[:120]}")
        if bk is None:
            raise RuntimeError(f"no feature backbone available: {err}")
        cfg = load_cfg(run)
        cfg["feat_backbone_resolved"] = bk
        save_cfg(run, cfg)
        if bk == "classifier":
            log("WARNING: using classifier features (no internet?). Metrics are still valid but "
                "less general than DINOv2; enable Internet in Kaggle settings for DINOv2.")
    else:
        ext = FeatureExtractor(bk, device, run)
    log(f"feature backbone: {bk}")
    _CACHE[key] = ext
    return ext


def featurize_file(run, npy_path, device=None, need_feats=True, need_probs=True):

    from .classifier import predict_probs
    device = device or get_device()
    base = npy_path[:-4]
    fp, pp = base + "_feats.npy", base + "_probs.npy"
    imgs = None
    if need_feats and not os.path.exists(fp):
        imgs = np.load(npy_path)
        log(f"features for {os.path.relpath(npy_path, run)} ({len(imgs)} imgs)")
        npsave(fp, get_extractor(run, device)(imgs).astype(np.float32))
    if need_probs and not os.path.exists(pp):
        imgs = np.load(npy_path) if imgs is None else imgs
        npsave(pp, predict_probs(imgs, run=run, device=device).astype(np.float32))
    feats = np.load(fp) if need_feats else None
    probs = np.load(pp) if need_probs else None
    return feats, probs

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/rdb/metrics.py

import numpy as np
import torch

def class_hist(probs, conf_thresh=0.5):
    pred = probs.argmax(1)
    conf = probs.max(1)
    ok = conf >= conf_thresh
    counts = np.bincount(pred[ok], minlength=10).astype(float)
    return counts, float(1 - ok.mean())


def bbse_correct(counts, confusion):
    p_pred = counts / max(counts.sum(), 1)
    sol, *_ = np.linalg.lstsq(confusion.T, p_pred, rcond=None)
    sol = np.clip(sol, 1e-6, None)
    return sol / sol.sum()


def composition_metrics(counts, p_ref, head_idx, tail_idx, smooth=0.5):
    n = counts.sum()
    p = (counts + smooth) / (n + smooth * len(counts))
    x, y = np.log(p_ref), np.log(p)
    A = np.vstack([x, np.ones_like(x)]).T
    (beta, alpha), *_ = np.linalg.lstsq(A, y, rcond=None)
    yhat = A @ np.array([beta, alpha])
    r2 = 1 - ((y - yhat) ** 2).sum() / max(((y - y.mean()) ** 2).sum(), 1e-12)
    tail_ref = p_ref[tail_idx].sum()
    return dict(
        beta=float(beta), beta_r2=float(r2),
        trr=float(p[tail_idx].sum() / tail_ref),
        head_share=float(p[head_idx].sum()),
        tail_share=float(p[tail_idx].sum()),
        kl=float(np.sum(p_ref * np.log(p_ref / p))),
        tv=float(0.5 * np.abs(p_ref - p).sum()),
        amplification=(p / p_ref).tolist(),
        p=p.tolist(),
    )

def _t(x, device):
    return torch.as_tensor(np.asarray(x), dtype=torch.float32, device=device)


def l2n(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)


def pairwise_sqdist(a, b):
    d = (a * a).sum(1, keepdim=True) + (b * b).sum(1)[None] - 2 * a @ b.T
    return d.clamp_min(0)


def knn_radii(x, k, device, chunk=2048):
    X = _t(x, device)
    out = []
    for i in range(0, len(X), chunk):
        d = pairwise_sqdist(X[i:i + chunk], X)
        d[torch.arange(d.shape[0]), torch.arange(i, i + d.shape[0])] = float("inf")
        out.append(d.topk(k, dim=1, largest=False).values[:, -1].sqrt().cpu())
    return torch.cat(out).numpy()


def knn_to_ref(q, ref, k, device, chunk=2048):
    Q, R = _t(q, device), _t(ref, device)
    out = []
    for i in range(0, len(Q), chunk):
        d = pairwise_sqdist(Q[i:i + chunk], R)
        out.append(d.topk(k, dim=1, largest=False).values[:, -1].sqrt().cpu())
    return torch.cat(out).numpy()


def prdc_rarity(real, fake, real_r, fake_r, k, device, real_groups=None, chunk=1024):

    R, Fk = _t(real, device), _t(fake, device)
    rr, fr = _t(real_r, device), _t(fake_r, device)
    prec_hits, dens, rar = [], [], []
    real_min_d = torch.full((len(R),), float("inf"), device=device)
    for i in range(0, len(Fk), chunk):
        d = pairwise_sqdist(Fk[i:i + chunk], R).sqrt()            # (f, r)
        inside = d <= rr[None]
        prec_hits.append(inside.any(1).cpu())
        dens.append(inside.sum(1).float().cpu())
        rad = torch.where(inside, rr[None].expand_as(d), torch.full_like(d, float("inf")))
        rar.append(rad.min(1).values.cpu())
        real_min_d = torch.minimum(real_min_d, d.min(0).values)
    precision = torch.cat(prec_hits).float().mean().item()
    density = (torch.cat(dens).sum() / (k * len(Fk))).item()
    rar = torch.cat(rar)
    rar = rar[torch.isfinite(rar)]
    covered = (real_min_d <= rr).cpu().numpy()
    coverage = float(covered.mean())

    rec_hits = []
    for i in range(0, len(R), chunk):
        d = pairwise_sqdist(R[i:i + chunk], Fk).sqrt()
        rec_hits.append((d <= fr[None]).any(1).cpu())
    recall = torch.cat(rec_hits).float().mean().item()
    out = dict(precision=precision, recall=recall, density=density, coverage=coverage,
               rarity=float(rar.mean().item()) if len(rar) else float("nan"))
    if real_groups is not None:
        for name, mask in real_groups.items():
            out[f"coverage_{name}"] = float(covered[mask].mean())
    return out


def density_occupancy(real_r, fake_to_real_r):
    lo, hi = np.quantile(real_r, [0.25, 0.75])
    hdo = float((fake_to_real_r < lo).mean())
    ldo = float((fake_to_real_r > hi).mean())
    return dict(hdo=hdo, ldo=ldo, dsi=hdo - ldo)


def frechet_distance(a, b):

    mu1, mu2 = a.mean(0), b.mean(0)
    s1, s2 = np.cov(a, rowvar=False), np.cov(b, rowvar=False)
    ev, U = np.linalg.eigh(s1)
    r1 = (U * np.sqrt(np.clip(ev, 0, None))) @ U.T
    tr_covmean = np.sqrt(np.clip(np.linalg.eigvalsh(r1 @ s2 @ r1), 0, None)).sum()
    return float(((mu1 - mu2) ** 2).sum() + np.trace(s1) + np.trace(s2) - 2 * tr_covmean)


def vendi_score(feats):
    X = l2n(feats.astype(np.float64))
    lam = np.linalg.eigvalsh(X.T @ X / len(X))
    lam = lam[lam > 1e-12]
    lam = lam / lam.sum()
    return float(np.exp(-(lam * np.log(lam)).sum()))



class Reference:
    def __init__(self, real_feats, real_labels, head_idx, tail_idx, k, device):
        self.raw = real_feats
        self.feats = l2n(real_feats)
        self.labels = real_labels
        self.k, self.device = k, device
        self.radii = knn_radii(self.feats, k, device)
        counts = np.bincount(real_labels, minlength=10)
        self.p_ref = counts / counts.sum()
        self.head_idx, self.tail_idx = head_idx, tail_idx
        self.groups = dict(head=np.isin(real_labels, head_idx), tail=np.isin(real_labels, tail_idx))


def evaluate_set(ref, fake_feats, fake_probs, confusion=None, conf_thresh=0.5, full=True):
    counts, unk = class_hist(fake_probs, conf_thresh)
    m = composition_metrics(counts, ref.p_ref, ref.head_idx, ref.tail_idx)
    m["unknown_rate"] = unk
    m["counts"] = counts.tolist()
    if confusion is not None:
        pc = bbse_correct(counts, confusion)
        mc = composition_metrics(pc * counts.sum(), ref.p_ref, ref.head_idx, ref.tail_idx)
        m["beta_bbse"], m["trr_bbse"], m["kl_bbse"] = mc["beta"], mc["trr"], mc["kl"]
    if not full:
        return m
    ff = l2n(fake_feats)
    f_to_r = knn_to_ref(ff, ref.feats, ref.k, ref.device)
    m.update(density_occupancy(ref.radii, f_to_r))
    fake_r = knn_radii(ff, ref.k, ref.device)
    m.update(prdc_rarity(ref.feats, ff, ref.radii, fake_r, ref.k, ref.device, ref.groups))
    m["fd"] = frechet_distance(ref.raw.astype(np.float64), fake_feats.astype(np.float64))
    m["vendi"] = vendi_score(fake_feats)
    return m

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/rdb/mitigation.py

import numpy as np
import torch

from .metrics import _t, l2n


def weighted_sample_without_replacement(w, n, rng):
    w = np.asarray(w, dtype=np.float64)
    pos = w > 0
    keys = np.full(len(w), -np.inf)
    u = rng.random(pos.sum())
    keys[pos] = np.log(u) / w[pos]
    return np.argsort(-keys)[:n]


def knn_real_fraction(pool_feats, real_feats, k, device, chunk=2048):
    P, R = _t(l2n(pool_feats), device), _t(l2n(real_feats), device)
    U = torch.cat([R, P])
    is_real = torch.cat([torch.ones(len(R), device=device), torch.zeros(len(P), device=device)])
    out = []
    for i in range(0, len(P), chunk):
        q = P[i:i + chunk]
        sim = q @ U.T                                         
        rows = torch.arange(len(q), device=device)
        sim[rows, len(R) + i + rows] = -float("inf")            
        nn = sim.topk(k, dim=1).indices
        out.append(is_real[nn].sum(1).cpu())
    return torch.cat(out).numpy()


def kdr_weights(pool_feats, real_feats, k=50, alpha=1.0, gamma=1.0, clip_q=0.005, device="cpu"):
    nr = knn_real_fraction(pool_feats, real_feats, k, device)
    ratio = (nr + alpha) / (k - nr + alpha) * (len(pool_feats) / len(real_feats))
    w = ratio ** gamma
    lo, hi = np.quantile(w, [clip_q, 1 - clip_q])
    return np.clip(w, lo, hi), nr


def intrinsic_dim_mle(dists):
    d = np.maximum(dists, 1e-10)
    inv = np.log(d[:, -1:] / d[:, :-1]).mean(1)
    return float(1.0 / np.mean(inv))


def flatten_weights(pool_feats, k=10, gamma=0.5, clip_q=0.01, device="cpu", chunk=2048):
    X = _t(l2n(pool_feats), device)
    D = []
    for i in range(0, len(X), chunk):
        d = ((X[i:i + chunk] * X[i:i + chunk]).sum(1, keepdim=True) + (X * X).sum(1)[None]
             - 2 * X[i:i + chunk] @ X.T).clamp_min(0)
        rows = torch.arange(d.shape[0], device=device)
        d[rows, i + rows] = float("inf")
        D.append(d.topk(k, dim=1, largest=False).values.sqrt().cpu())
    D = torch.cat(D).numpy()
    m = intrinsic_dim_mle(D)
    logw = gamma * m * np.log(D[:, -1])        
    logw -= logw.max()
    w = np.exp(logw)
    lo, hi = np.quantile(w, [clip_q, 1 - clip_q])
    return np.clip(w, lo, hi), m


def select_training_set(strategy, n, pool_feats, pool_probs, real_feats, real_labels, params,
                        conf_thresh, seed, device):
    rng = np.random.default_rng(seed)
    N = len(pool_probs)
    info = dict(strategy=strategy, pool_size=N, n=n)
    if strategy == "random":
        return rng.choice(N, n, replace=False), 0, info
    if strategy == "realmix":
        n_real = int(round(params.get("real_frac", 0.5) * n))
        info["n_real"] = n_real
        return rng.choice(N, n - n_real, replace=False), n_real, info
    if strategy == "kdr":
        w, nr = kdr_weights(pool_feats, real_feats, k=params.get("k", 50), alpha=params.get("alpha", 1.0),
                            gamma=params.get("gamma", 1.0), clip_q=params.get("clip_q", 0.005), device=device)
        idx = weighted_sample_without_replacement(w, n, rng)
        info.update(w_min=float(w.min()), w_max=float(w.max()), w_mean=float(w.mean()),
                    real_nbr_frac_mean=float(nr.mean() / params.get("k", 50)),
                    ess=float(w.sum() ** 2 / (w ** 2).sum()))
        return idx, 0, info
    if strategy == "flatten":
        w, m = flatten_weights(pool_feats, k=params.get("k", 10), gamma=params.get("gamma", 0.5),
                               clip_q=params.get("clip_q", 0.01), device=device)
        info.update(intrinsic_dim=m, ess=float(w.sum() ** 2 / (w ** 2).sum()))
        return weighted_sample_without_replacement(w, n, rng), 0, info
    if strategy == "oracle":
        p_ref = np.bincount(real_labels, minlength=10) / len(real_labels)
        pred, conf = pool_probs.argmax(1), pool_probs.max(1)
        ok = conf >= conf_thresh
        target = np.floor(p_ref * n).astype(int)
        target[np.argsort(-(p_ref * n - target))[: n - target.sum()]] += 1
        chosen, short = [], 0
        for c in range(10):
            cand = np.where(ok & (pred == c))[0]
            take = min(target[c], len(cand))
            short += target[c] - take
            chosen.append(rng.choice(cand, take, replace=False))
        chosen = np.concatenate(chosen)
        if short > 0: 
            rest = np.setdiff1d(np.where(ok)[0], chosen)
            if len(rest) < short:
                rest = np.setdiff1d(np.arange(N), chosen)
            chosen = np.concatenate([chosen, rng.choice(rest, short, replace=False)])
        info["shortfall"] = int(short)
        return chosen, 0, info
    raise ValueError(strategy)

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/rdb/pipeline.py

import argparse
import math
import os
import subprocess
import sys
import threading
import time

import numpy as np

from .common import fmt_secs, jload, jsave, load_cfg, log, npsave, save_cfg, seed_all

def gen0_dir(run):
    return os.path.join(run, "gen0")


def chain_dir(run, name, g):
    return os.path.join(run, "chains", name, f"gen{g}")


def n_real(run):
    return len(np.load(os.path.join(run, "data", "real_labels.npy")))


def gen0_pool_size(run, cfg):
    mult = max(cfg["chains"][c]["pool_mult"] for c in cfg["run_chains"])
    return int(math.ceil(mult * n_real(run)))


def eval_file(run, cfg, name, g):
    """Image file holding (at least) N unfiltered samples from the generation-g model."""
    if g == 0:
        return os.path.join(gen0_dir(run), "pool.npy")
    d = chain_dir(run, name, g)
    return os.path.join(d, "samples.npy" if g == cfg["gens"] else "pool.npy")


def stage_prepare(a):
    from .config import make_config
    from .data import prepare_data
    os.makedirs(a.run, exist_ok=True)
    cpath = os.path.join(a.run, "config.json")
    if os.path.exists(cpath):
        cfg = load_cfg(a.run)
        if a.preset and cfg["preset"] != a.preset:
            log(f"NOTE: run already configured with preset '{cfg['preset']}' (ignoring '{a.preset}')")
    else:
        over = {}
        if a.chains:
            over["run_chains"] = a.chains.split(",")
        if a.model:
            over["model"] = a.model
        cfg = make_config(a.preset or "full", **over)
        save_cfg(a.run, cfg)
    prepare_data(a.run, cfg, source=a.source, cifar_root=a.cifar_root)


def stage_classifier(a):
    from .classifier import train_classifier
    train_classifier(a.run, load_cfg(a.run))


def stage_features(a):
    from .features import featurize_file
    featurize_file(a.run, os.path.join(a.run, "data", "real_train.npy"), need_probs=True)


def stage_train_gen0(a):
    from .config import MODEL_CFGS
    from .diffusion import train_diffusion
    cfg = load_cfg(a.run)
    seed_all(cfg["seed"])
    train_diffusion(os.path.join(a.run, "data", "real_train.npy"), gen0_dir(a.run), MODEL_CFGS[cfg["model"]],
                    steps=cfg["gen0_steps"], batch=cfg["gen0_batch"], lr=cfg["gen0_lr"], warmup=cfg["warmup"],
                    ema_decay=cfg["ema"], grad_clip=cfg["grad_clip"], seed=cfg["seed"], T=cfg["T"],
                    schedule=cfg["schedule"], meta=dict(stage="gen0"))


def stage_sample(a):
    from .common import get_device
    from .diffusion import ddim_sample, load_model
    cfg = load_cfg(a.run)
    out = a.out if a.nshards == 1 else f"{a.out[:-4]}.part{a.shard}.npy"
    if os.path.exists(a.out) or os.path.exists(out):
        log(f"{out} exists - skipping")
        return
    per = int(math.ceil(a.n / a.nshards))
    n = min(per, a.n - a.shard * per)
    device = get_device()
    model, _ = load_model(a.ckpt, device)
    log(f"sampling {n} images (shard {a.shard}/{a.nshards}) from {os.path.relpath(a.ckpt, a.run)}")
    imgs = ddim_sample(model, n, steps=cfg["ddim_steps"], T=cfg["T"], schedule=cfg["schedule"],
                       batch=cfg["sample_batch"], seed=a.seed * 1000 + a.shard, noise_scale=cfg["noise_scale"],
                       device=device)
    npsave(out, imgs)


def stage_merge(a):
    if os.path.exists(a.out):
        return
    parts = [f"{a.out[:-4]}.part{i}.npy" for i in range(a.nshards)]
    npsave(a.out, np.concatenate([np.load(p) for p in parts]))
    for p in parts:
        os.remove(p)
    log(f"merged {a.out}")


def stage_featurize(a):
    from .features import featurize_file
    featurize_file(a.run, a.npy)


def stage_chain(a):
    from .common import get_device
    from .config import MODEL_CFGS
    from .diffusion import ddim_sample, load_model, train_diffusion
    from .features import featurize_file
    from .mitigation import select_training_set
    cfg = load_cfg(a.run)
    name = a.name
    params = cfg["chains"][name]
    device = get_device()
    N = n_real(a.run)
    real_imgs = np.load(os.path.join(a.run, "data", "real_train.npy"))
    real_labels = np.load(os.path.join(a.run, "data", "real_labels.npy"))
    real_feats, _ = featurize_file(a.run, os.path.join(a.run, "data", "real_train.npy"), device)
    chain_idx = sorted(cfg["chains"]).index(name)
    prev_ckpt = os.path.join(gen0_dir(a.run), "ema.pt")
    t_chain = time.time()
    for g in range(1, cfg["gens"] + 1):
        d = chain_dir(a.run, name, g)
        os.makedirs(d, exist_ok=True)
        if not os.path.exists(os.path.join(d, "train.npy")):
            if g == 1:
                pool_path = os.path.join(gen0_dir(a.run), "pool.npy")
                limit = int(math.ceil(params["pool_mult"] * N))
            else:
                pool_path = os.path.join(chain_dir(a.run, name, g - 1), "pool.npy")
                limit = None
            pool_feats, pool_probs = featurize_file(a.run, pool_path, device)
            pool = np.load(pool_path, mmap_mode="r")
            if limit is not None:
                pool_feats, pool_probs = pool_feats[:limit], pool_probs[:limit]
            sel_seed = cfg["seed"] * 1000 + g * 100 + chain_idx
            idx, n_add_real, info = select_training_set(
                params["strategy"], N, pool_feats, pool_probs, real_feats, real_labels, params,
                cfg["conf_thresh"], sel_seed, device)
            imgs = np.asarray(pool[np.sort(idx)])
            src = np.zeros(len(imgs), np.int8)
            if n_add_real:
                ridx = np.random.default_rng(sel_seed + 7).choice(len(real_imgs), n_add_real, replace=False)
                imgs = np.concatenate([imgs, real_imgs[ridx]])
                src = np.concatenate([src, np.ones(n_add_real, np.int8)])
                np.save(os.path.join(d, "train_real_idx.npy"), ridx)
            np.save(os.path.join(d, "train_idx.npy"), np.sort(idx))
            np.save(os.path.join(d, "train_is_real.npy"), src)
            info.update(pool=os.path.relpath(pool_path, a.run), pool_used=len(pool_probs), n_train=len(imgs))
            jsave(info, os.path.join(d, "train_meta.json"))
            npsave(os.path.join(d, "train.npy"), imgs)
            log(f"[{name}] gen{g}: training set {len(imgs)} images  {info}")

        ckpt = train_diffusion(os.path.join(d, "train.npy"), d, MODEL_CFGS[cfg["model"]],
                               steps=cfg["ft_steps"], batch=cfg["ft_batch"], lr=cfg["ft_lr"],
                               warmup=cfg["warmup"], ema_decay=cfg["ema"], grad_clip=cfg["grad_clip"],
                               init=prev_ckpt, seed=cfg["seed"] + g * 31 + chain_idx, T=cfg["T"],
                               schedule=cfg["schedule"], meta=dict(chain=name, gen=g))

        last = g == cfg["gens"]
        out = os.path.join(d, "samples.npy" if last else "pool.npy")
        n = N if last else int(math.ceil(params["pool_mult"] * N))
        if not os.path.exists(out):
            model, _ = load_model(ckpt, device)
            log(f"[{name}] gen{g}: sampling {n} images")
            imgs = ddim_sample(model, n, steps=cfg["ddim_steps"], T=cfg["T"], schedule=cfg["schedule"],
                               batch=cfg["sample_batch"], seed=cfg["seed"] * 1000 + g * 100 + chain_idx + 50,
                               noise_scale=cfg["noise_scale"], device=device)
            npsave(out, imgs)
            del model
        featurize_file(a.run, out, device)
        prev_ckpt = ckpt
    log(f"[{name}] chain complete ({fmt_secs(time.time() - t_chain)})")


def stage_evaluate(a):
    from .evaluate import evaluate_run
    evaluate_run(a.run)


def run_jobs(run, jobs):
    """Run jobs in parallel; each job is (tag, argv, extra_env). Streams prefixed output."""
    os.makedirs(os.path.join(run, "logs"), exist_ok=True)
    procs = []
    for tag, argv, env in jobs:
        logf = open(os.path.join(run, "logs", f"{tag}.log"), "a")
        pkg_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
        pypath = pkg_root + (os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")
        p = subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                             env={**os.environ, **env, "PYTHONUNBUFFERED": "1", "PYTHONPATH": pypath})

        def pump(p=p, tag=tag, logf=logf):
            for line in p.stdout:
                logf.write(line); logf.flush()
                print(f"[{tag}] {line}", end="", flush=True)
        th = threading.Thread(target=pump, daemon=True)
        th.start()
        procs.append((tag, p, th))
    failed = []
    for tag, p, th in procs:
        p.wait(); th.join()
        if p.returncode != 0:
            failed.append(tag)
    if failed:
        raise RuntimeError(f"job(s) failed: {failed}; see {run}/logs/")


def _py(*args):
    return [sys.executable, "-m", "rdb.pipeline", *map(str, args)]


def stage_all(a):
    import torch
    t0 = time.time()
    ngpu = torch.cuda.device_count()
    stage_prepare(a)
    cfg = load_cfg(a.run)
    stages = a.stages.split(",") if a.stages else ["setup", "gen0", "pool", "chains", "evaluate"]
    log(f"GPUs: {ngpu}  preset: {cfg['preset']}  chains: {cfg['run_chains']}  stages: {stages}")
    gpu_env = (lambda i: {"CUDA_VISIBLE_DEVICES": str(i)}) if ngpu else (lambda i: {})

    if "setup" in stages:
        run_jobs(a.run, [("classifier", _py("classifier", "--run", a.run), gpu_env(0))])
        run_jobs(a.run, [("features", _py("features", "--run", a.run), gpu_env(0))])

    if "gen0" in stages:
        if ngpu >= 2:
            argv = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={ngpu}",
                    "-m", "rdb.pipeline", "train-gen0", "--run", a.run]
        else:
            argv = _py("train-gen0", "--run", a.run)
        run_jobs(a.run, [("gen0", argv, {})])

    if "pool" in stages:
        pool = os.path.join(gen0_dir(a.run), "pool.npy")
        ck = os.path.join(gen0_dir(a.run), "ema.pt")
        npool = gen0_pool_size(a.run, cfg)
        k = max(1, ngpu)
        if not os.path.exists(pool):
            run_jobs(a.run, [(f"pool{i}", _py("sample", "--run", a.run, "--ckpt", ck, "--n", npool, "--out", pool,
                                              "--shard", i, "--nshards", k, "--seed", cfg["seed"] + 1), gpu_env(i))
                             for i in range(k)])
            stage_merge(argparse.Namespace(out=pool, nshards=k))
        run_jobs(a.run, [("pool-feat", _py("featurize", "--run", a.run, "--npy", pool), gpu_env(0))])

    if "chains" in stages:
        k = max(1, ngpu)
        queues = [[] for _ in range(k)]
        for i, name in enumerate(cfg["run_chains"]):
            queues[i % k].append(name)
        jobs = []
        for i, q in enumerate(queues):
            if not q:
                continue
            steps = " && ".join(" ".join(_py("chain", "--run", a.run, "--name", n)) for n in q)
            jobs.append((f"gpu{i}", ["bash", "-c", steps], gpu_env(i)))
        log("chain queues: " + "; ".join(f"gpu{i}: {q}" for i, q in enumerate(queues)))
        run_jobs(a.run, jobs)

    if "evaluate" in stages:
        run_jobs(a.run, [("evaluate", _py("evaluate", "--run", a.run), gpu_env(0))])
    log(f"ALL DONE in {fmt_secs(time.time() - t0)}  ->  {os.path.join(a.run, 'results')}")


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("stage")
    p.add_argument("--run", required=True)
    p.add_argument("--preset", default=None)
    p.add_argument("--chains", default=None, help="comma list overriding run_chains (new runs only)")
    p.add_argument("--model", default=None, help="base | small | large (new runs only)")
    p.add_argument("--stages", default=None, help="for 'all': subset of setup,gen0,pool,chains,evaluate")
    p.add_argument("--source", default="auto", help="auto | fake")
    p.add_argument("--cifar_root", default="./cifar_data")
    p.add_argument("--ckpt"); p.add_argument("--n", type=int); p.add_argument("--out")
    p.add_argument("--shard", type=int, default=0); p.add_argument("--nshards", type=int, default=1)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--npy"); p.add_argument("--name")
    a = p.parse_args(argv)
    a.run = os.path.abspath(a.run)
    fn = {"all": stage_all, "prepare": stage_prepare, "classifier": stage_classifier,
          "features": stage_features, "train-gen0": stage_train_gen0, "sample": stage_sample,
          "merge": stage_merge, "featurize": stage_featurize, "chain": stage_chain,
          "evaluate": stage_evaluate}[a.stage]
    fn(a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/rdb/evaluate.py

import os

import numpy as np

from .common import get_device, jload, jsave, load_cfg, log, save_grid
from .config import CIFAR10_CLASSES
from .metrics import Reference, composition_metrics, evaluate_set, class_hist


CHAIN_COLORS = {"baseline": "#2a78d6", "kdr": "#eb6834", "oracle": "#1baf7a",
                "realmix": "#eda100", "flatten": "#e87ba4"}
CHAIN_LABELS = {"baseline": "Baseline (random)", "kdr": "KDR-IR (proposed)", "oracle": "Oracle (labels)",
                "realmix": "Real-mix 50%", "flatten": "Density flatten"}
GEN_RAMP = ["#86b6ef", "#3987e5", "#1c5cab", "#104281"]     
REF_GRAY = "#898781"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e1e0d9"

KEY_METRICS = ["beta", "beta_bbse", "trr", "trr_bbse", "kl", "tv", "head_share", "unknown_rate",
               "hdo", "ldo", "dsi", "rarity", "precision", "recall", "density", "coverage",
               "coverage_head", "coverage_tail", "fd", "vendi"]


def _style(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#c3c2b7")
    ax.tick_params(colors=INK2, labelsize=8)
    ax.grid(axis="y", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)


def mitigation_rate(base_val, mit_val, ideal):
    db = abs(base_val - ideal)
    if db < 1e-9:
        return float("nan")
    return float(1 - abs(mit_val - ideal) / db)


def evaluate_run(run):
    import matplotlib
    matplotlib.use("Agg")
    cfg = load_cfg(run)
    device = get_device()
    res_dir = os.path.join(run, "results")
    fig_dir, grid_dir = os.path.join(res_dir, "figures"), os.path.join(res_dir, "grids")
    os.makedirs(fig_dir, exist_ok=True); os.makedirs(grid_dir, exist_ok=True)
    G = cfg["gens"]
    head_idx = [CIFAR10_CLASSES.index(c) for c in cfg["head"]]
    tail_idx = [CIFAR10_CLASSES.index(c) for c in cfg["tail"]]
    d = os.path.join(run, "data")
    real_labels = np.load(os.path.join(d, "real_labels.npy"))
    real_feats = np.load(os.path.join(d, "real_train_feats.npy"))
    real_probs = np.load(os.path.join(d, "real_train_probs.npy"))
    N = len(real_labels)
    conf_path = os.path.join(run, "classifier", "confusion_test.npy")
    confusion = np.load(conf_path) if os.path.exists(conf_path) else None
    ref = Reference(real_feats, real_labels, head_idx, tail_idx, cfg["knn_k"], device)
    th = cfg["conf_thresh"]

    def load_set(path):
        base = path[:-4]
        return (np.load(base + "_feats.npy")[:N], np.load(base + "_probs.npy")[:N])

    results = dict(config=dict(preset=cfg["preset"], gens=G, chains=cfg["run_chains"],
                               feat_backbone=cfg.get("feat_backbone_resolved"), N=N, k=cfg["knn_k"]),
                   p_ref=ref.p_ref.tolist(), classes=CIFAR10_CLASSES, head=cfg["head"], tail=cfg["tail"])
    results["real_classifier"] = evaluate_set(ref, None, real_probs, confusion, th, full=False)
    log("evaluating gen0")
    g0 = evaluate_set(ref, *load_set(os.path.join(run, "gen0", "pool.npy")), confusion, th)
    results["gen0"] = g0
    results["chains"], results["train_sets"] = {}, {}
    for name in cfg["run_chains"]:
        results["chains"][name] = {"0": g0}
        results["train_sets"][name] = {}
        for g in range(1, G + 1):
            dd = os.path.join(run, "chains", name, f"gen{g}")
            f = os.path.join(dd, "samples.npy" if g == G else "pool.npy")
            if not os.path.exists(f[:-4] + "_probs.npy"):
                log(f"  {name} gen{g}: missing, skipped")
                continue
            log(f"evaluating {name} gen{g}")
            results["chains"][name][str(g)] = evaluate_set(ref, *load_set(f), confusion, th)

            meta = jload(os.path.join(dd, "train_meta.json"))
            pool_probs = np.load(os.path.join(run, meta["pool"])[:-4] + "_probs.npy")
            idx = np.load(os.path.join(dd, "train_idx.npy"))
            counts, _ = class_hist(pool_probs[idx], th)
            rp = os.path.join(dd, "train_real_idx.npy")
            if os.path.exists(rp):
                counts = counts + np.bincount(real_labels[np.load(rp)], minlength=10)
            pool_counts, _ = class_hist(pool_probs[:meta["pool_used"]], th)
            results["train_sets"][name][str(g)] = dict(
                train=composition_metrics(counts, ref.p_ref, head_idx, tail_idx),
                pool=composition_metrics(pool_counts, ref.p_ref, head_idx, tail_idx), meta=meta)


    eff = {}
    base = results["chains"].get("baseline", {}).get(str(G))
    if base:
        for name in cfg["run_chains"]:
            m = results["chains"][name].get(str(G))
            if name == "baseline" or not m:
                continue
            eff[name] = dict(
                bmr_beta=mitigation_rate(base["beta"], m["beta"], 1.0),
                bmr_trr=mitigation_rate(base["trr"], m["trr"], 1.0),
                bmr_kl=mitigation_rate(base["kl"], m["kl"], 0.0),
                bmr_dsi=mitigation_rate(base["dsi"], m["dsi"], 0.0),
                d_beta=m["beta"] - base["beta"], d_trr=m["trr"] - base["trr"],
                d_fd=m["fd"] - base["fd"], d_cov_tail=m["coverage_tail"] - base["coverage_tail"])
    results["effectiveness"] = eff
    jsave(results, os.path.join(res_dir, "metrics.json"))
    _write_tables(results, res_dir, G)
    _figures(results, fig_dir, G)
    _grids(run, cfg, results, grid_dir, G, N)
    log(f"results written to {res_dir}")
    print(open(os.path.join(res_dir, "summary.md")).read())
    return results


def _write_tables(R, res_dir, G):
    with open(os.path.join(res_dir, "metrics.csv"), "w") as f:
        f.write("chain,gen," + ",".join(KEY_METRICS) + "\n")
        for name, gens in R["chains"].items():
            for g, m in sorted(gens.items(), key=lambda kv: int(kv[0])):
                f.write(f"{name},{g}," + ",".join(f"{m.get(k, float('nan')):.5f}" for k in KEY_METRICS) + "\n")
    with open(os.path.join(res_dir, "class_distribution.csv"), "w") as f:
        f.write("chain,gen," + ",".join(R["classes"]) + "\n")
        f.write("original,-," + ",".join(f"{p:.5f}" for p in R["p_ref"]) + "\n")
        for name, gens in R["chains"].items():
            for g, m in sorted(gens.items(), key=lambda kv: int(kv[0])):
                f.write(f"{name},{g}," + ",".join(f"{p:.5f}" for p in m["p"]) + "\n")
    L = ["# Results summary", "",
         f"Feature backbone: `{R['config']['feat_backbone']}` | N = {R['config']['N']} | k = {R['config']['k']}", "",
         "Ideal values: beta = 1, TRR = 1, KL = 0, HDO = LDO = 0.25, DSI = 0.", "",
         "| chain | gen | beta | beta (BBSE) | TRR | KL | DSI | rarity | precision | recall | coverage (tail) | FD | Vendi |",
         "|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for name, gens in R["chains"].items():
        for g, m in sorted(gens.items(), key=lambda kv: int(kv[0])):
            L.append(f"| {name} | {g} | {m['beta']:.3f} | {m.get('beta_bbse', float('nan')):.3f} | {m['trr']:.3f} | "
                     f"{m['kl']:.4f} | {m['dsi']:+.3f} | {m['rarity']:.3f} | {m['precision']:.3f} | "
                     f"{m['recall']:.3f} | {m['coverage_tail']:.3f} | {m['fd']:.1f} | {m['vendi']:.2f} |")
    if R["effectiveness"]:
        L += ["", f"## Mitigation effectiveness at Gen {G} (vs baseline)", "",
              "Bias Mitigation Rate (BMR) = 1 - |mitigated - ideal| / |baseline - ideal|  "
              "(1 = all of the baseline's excess bias removed, 0 = no change, < 0 = worse).", "",
              "| chain | BMR(beta) | BMR(TRR) | BMR(KL) | BMR(DSI) | d beta | d TRR | d FD | d coverage(tail) |",
              "|---|---|---|---|---|---|---|---|---|"]
        for name, e in R["effectiveness"].items():
            L.append(f"| {name} | {e['bmr_beta']:.2f} | {e['bmr_trr']:.2f} | {e['bmr_kl']:.2f} | {e['bmr_dsi']:.2f} | "
                     f"{e['d_beta']:+.3f} | {e['d_trr']:+.3f} | {e['d_fd']:+.1f} | {e['d_cov_tail']:+.3f} |")
    L += ["", "## Training-set composition after selection (classifier-measured)", "",
          "| chain | gen | pool beta | pool TRR | train beta | train TRR |", "|---|---|---|---|---|---|"]
    for name, gens in R["train_sets"].items():
        for g, t in sorted(gens.items()):
            L.append(f"| {name} | {g} | {t['pool']['beta']:.3f} | {t['pool']['trr']:.3f} | "
                     f"{t['train']['beta']:.3f} | {t['train']['trr']:.3f} |")
    with open(os.path.join(res_dir, "summary.md"), "w") as f:
        f.write("\n".join(L) + "\n")


def _figures(R, fig_dir, G):
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family": "sans-serif", "font.size": 9, "axes.titlesize": 10,
                         "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.labelcolor": INK2})
    chains = list(R["chains"])
    gens = list(range(G + 1))


    panels = [("beta", "Density amplification β  (ideal 1)", 1.0), ("trr", "Tail retention ratio  (ideal 1)", 1.0),
              ("kl", "KL(original ‖ generated)  (ideal 0)", 0.0), ("dsi", "Density skew index  (ideal 0)", 0.0),
              ("coverage_tail", "Coverage of tail classes  (↑)", None), ("fd", "FD (DINOv2)  (↓)", None)]
    fig, axes = plt.subplots(2, 3, figsize=(12, 6.4))
    for ax, (key, title, ideal) in zip(axes.flat, panels):
        _style(ax)
        if ideal is not None:
            ax.axhline(ideal, color=REF_GRAY, lw=1, ls="--", zorder=1)
        for name in chains:
            ys = [R["chains"][name].get(str(g), {}).get(key, np.nan) for g in gens]
            ax.plot(gens, ys, color=CHAIN_COLORS.get(name, INK2), lw=2, marker="o", ms=6,
                    markeredgecolor="#fcfcfb", markeredgewidth=1.5, label=CHAIN_LABELS.get(name, name), zorder=3)
        ax.set_title(title, loc="left")
        ax.set_xticks(gens, [f"Gen {g}" for g in gens])
    h, l = axes.flat[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=len(chains), frameon=False, fontsize=9)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    fig.savefig(os.path.join(fig_dir, "metric_trajectories.png"), dpi=160, facecolor="#fcfcfb")
    plt.close(fig)

    order = np.argsort(-np.asarray(R["p_ref"]))
    names = [R["classes"][i] for i in order]
    x = np.arange(10)
    for name in chains:
        fig, ax = plt.subplots(figsize=(12, 4))
        _style(ax)
        series = [("Original split", np.asarray(R["p_ref"]), REF_GRAY)] + \
                 [(f"Gen {g}", np.asarray(R["chains"][name][str(g)]["p"]), GEN_RAMP[g % len(GEN_RAMP)])
                  for g in gens if str(g) in R["chains"][name]]
        w = 0.8 / len(series)
        for j, (lab, p, col) in enumerate(series):
            ax.bar(x + (j - (len(series) - 1) / 2) * w, p[order], width=w * 0.92, color=col, label=lab)
        ax.set_yscale("log")
        ax.set_xticks(x, names)
        ax.set_ylabel("share of samples (log)")
        tail_start = sum(1 for c in names if c in R["head"])
        ax.axvline(tail_start - 0.5, color="#c3c2b7", lw=1)
        ax.text(tail_start - 0.4, ax.get_ylim()[1], "tail →", va="top", fontsize=8, color=INK2)
        ax.set_title(f"Class composition across generations - {CHAIN_LABELS.get(name, name)}", loc="left")
        ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, f"class_distribution_{name}.png"), dpi=160, facecolor="#fcfcfb")
        plt.close(fig)

    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("div", ["#104281", "#3987e5", "#f0efec", "#e66767", "#a32b2b"])
    rows, labels = [], []
    for name in chains:
        for g in gens[1:]:
            if str(g) in R["chains"][name]:
                rows.append(np.log2(np.asarray(R["chains"][name][str(g)]["amplification"])[order]))
                labels.append(f"{name} · Gen {g}")
    rows.insert(0, np.log2(np.asarray(R["gen0"]["amplification"])[order])); labels.insert(0, "Gen 0")
    M = np.vstack(rows)
    lim = max(1.0, np.nanmax(np.abs(M)))
    fig, ax = plt.subplots(figsize=(10, 0.42 * len(rows) + 1.4))
    im = ax.imshow(M, cmap=cmap, vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(x, names); ax.set_yticks(range(len(labels)), labels)
    ax.tick_params(labelsize=8, colors=INK2, length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i, j]:+.1f}", ha="center", va="center", fontsize=7,
                    color="#ffffff" if abs(M[i, j]) > 0.6 * lim else INK)
    cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
    cb.set_label("log2(p_generated / p_original)", color=INK2, fontsize=8)
    cb.outline.set_visible(False)
    ax.set_title("Per-class amplification (red = over-produced, blue = under-produced)", loc="left")
    fig.tight_layout()
    fig.savefig(os.path.join(fig_dir, "amplification_heatmap.png"), dpi=160, facecolor="#fcfcfb")
    plt.close(fig)


    sel = [n for n in chains if "1" in R["train_sets"].get(n, {})]
    if sel:
        fig, ax = plt.subplots(figsize=(12, 4))
        _style(ax)
        series = [("Original split", np.asarray(R["p_ref"]), REF_GRAY),
                  ("Gen-0 samples (pool)", np.asarray(R["train_sets"][sel[0]]["1"]["pool"]["p"]), GEN_RAMP[0])]
        for n in sel:
            if n == "baseline":
                continue
            series.append((f"{CHAIN_LABELS.get(n, n)} selection", np.asarray(R["train_sets"][n]["1"]["train"]["p"]),
                           CHAIN_COLORS.get(n, INK2)))
        w = 0.8 / len(series)
        for j, (lab, p, col) in enumerate(series):
            ax.bar(x + (j - (len(series) - 1) / 2) * w, p[order], width=w * 0.92, color=col, label=lab)
        ax.set_yscale("log"); ax.set_xticks(x, names); ax.set_ylabel("share (log)")
        ax.set_title("How each mechanism re-composes the Gen-1 training set", loc="left")
        ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, "selection_mechanism_gen1.png"), dpi=160, facecolor="#fcfcfb")
        plt.close(fig)


def _grids(run, cfg, R, grid_dir, G, N):
    tail_idx = [CIFAR10_CLASSES.index(c) for c in cfg["tail"]]
    th = cfg["conf_thresh"]
    sets = [("gen0", os.path.join(run, "gen0", "pool.npy"))]
    for name in R["chains"]:
        for g in range(1, G + 1):
            f = os.path.join(run, "chains", name, f"gen{g}", "samples.npy" if g == G else "pool.npy")
            if os.path.exists(f):
                sets.append((f"{name}_gen{g}", f))
    for tag, f in sets:
        imgs = np.load(f, mmap_mode="r")
        n = min(N, len(imgs))
        pick = np.random.default_rng(0).choice(n, min(100, n), replace=False)
        save_grid(np.asarray(imgs[np.sort(pick)]), os.path.join(grid_dir, f"random_{tag}.png"), nrow=10)
        probs = np.load(f[:-4] + "_probs.npy")[:n]
        rows = []
        for c in tail_idx: 
            ok = np.where((probs.argmax(1) == c) & (probs.max(1) >= th))[0]
            ok = ok[np.argsort(-probs[ok, c])][:10]
            row = [np.asarray(imgs[i]) for i in ok] + [np.full((32, 32, 3), 255, np.uint8)] * (10 - len(ok))
            rows.extend(row)
        save_grid(np.stack(rows), os.path.join(grid_dir, f"tail_gallery_{tag}.png"), nrow=10)

In [ ]:
%%writefile /kaggle/working/recursive-density-bias/app.py

import argparse
import io
import os
import tempfile
import zipfile

import numpy as np
from PIL import Image

from rdb.common import get_device, jload, load_cfg
from rdb.config import CIFAR10_CLASSES
from rdb.evaluate import CHAIN_COLORS, CHAIN_LABELS, GEN_RAMP, REF_GRAY, INK2, GRID

RUN = None
DEVICE = None
_MODELS, _STATE = {}, {}

def _ref():
    if "ref" not in _STATE:
        from rdb.metrics import Reference
        cfg = load_cfg(RUN)
        d = os.path.join(RUN, "data")
        _STATE["cfg"] = cfg
        _STATE["real_labels"] = np.load(os.path.join(d, "real_labels.npy"))
        _STATE["real_feats"] = np.load(os.path.join(d, "real_train_feats.npy"))
        cp = os.path.join(RUN, "classifier", "confusion_test.npy")
        _STATE["confusion"] = np.load(cp) if os.path.exists(cp) else None
        _STATE["ref"] = Reference(_STATE["real_feats"], _STATE["real_labels"],
                                  [CIFAR10_CLASSES.index(c) for c in cfg["head"]],
                                  [CIFAR10_CLASSES.index(c) for c in cfg["tail"]], cfg["knn_k"], DEVICE)
    return _STATE["ref"]


def model_choices():
    cfg = load_cfg(RUN)
    out = []
    if os.path.exists(os.path.join(RUN, "gen0", "ema.pt")):
        out.append("gen0")
    for name in cfg["run_chains"]:
        for g in range(1, cfg["gens"] + 1):
            if os.path.exists(os.path.join(RUN, "chains", name, f"gen{g}", "ema.pt")):
                out.append(f"{name}/gen{g}")
    return out


def ckpt_path(choice):
    if choice == "gen0":
        return os.path.join(RUN, "gen0", "ema.pt")
    name, g = choice.split("/")
    return os.path.join(RUN, "chains", name, g, "ema.pt")


def grid_image(imgs, nrow=10, scale=3):
    n = len(imgs)
    ncol = min(nrow, n)
    nr = (n + ncol - 1) // ncol
    canvas = np.full((nr * 34 + 2, ncol * 34 + 2, 3), 255, np.uint8)
    for i, im in enumerate(imgs):
        r, c = divmod(i, ncol)
        canvas[2 + r * 34: 2 + r * 34 + 32, 2 + c * 34: 2 + c * 34 + 32] = im
    return Image.fromarray(canvas).resize((canvas.shape[1] * scale, canvas.shape[0] * scale), Image.NEAREST)


def composition_plot(series, title):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ref = _ref()
    order = np.argsort(-ref.p_ref)
    x = np.arange(10)
    fig, ax = plt.subplots(figsize=(10.5, 3.6))
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
    w = 0.8 / len(series)
    for j, (lab, p, col) in enumerate(series):
        ax.bar(x + (j - (len(series) - 1) / 2) * w, np.asarray(p)[order], width=w * 0.92, color=col, label=lab)
    ax.set_yscale("log")
    ax.set_xticks(x, [CIFAR10_CLASSES[i] for i in order], fontsize=8)
    ax.tick_params(colors=INK2)
    ax.set_ylabel("share (log)")
    ax.set_title(title, loc="left", fontsize=10, fontweight="bold")
    ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    fig.tight_layout()
    return fig


def measure(imgs, full=True):
    from rdb.features import get_extractor
    from rdb.classifier import predict_probs
    from rdb.metrics import evaluate_set
    ref = _ref()
    cfg = _STATE["cfg"]
    probs = predict_probs(imgs, run=RUN, device=DEVICE)
    feats = get_extractor(RUN, DEVICE)(imgs) if full else None
    m = evaluate_set(ref, feats, probs, _STATE["confusion"], cfg["conf_thresh"], full=full)
    return m, feats, probs


def metrics_table(m):
    rows = [("β  density amplification (ideal 1)", m["beta"]), ("β, confusion-corrected (BBSE)", m.get("beta_bbse")),
            ("TRR tail retention (ideal 1)", m["trr"]), ("KL(original ‖ generated) (ideal 0)", m["kl"]),
            ("head-class share (original %.3f)" % float(_ref().p_ref[_ref().head_idx].sum()), m["head_share"]),
            ("unrecognisable share", m["unknown_rate"])]
    for k, lab in [("hdo", "HDO high-density occupancy (ideal 0.25)"), ("ldo", "LDO low-density occupancy (ideal 0.25)"),
                   ("dsi", "DSI density skew index (ideal 0)"), ("rarity", "mean rarity score"),
                   ("precision", "precision"), ("recall", "recall"), ("coverage", "coverage"),
                   ("coverage_tail", "coverage of tail classes"), ("fd", "FD (feature space)"), ("vendi", "Vendi score")]:
        if k in m:
            rows.append((lab, m[k]))
    return [[a, None if b is None else round(float(b), 4)] for a, b in rows]


def load_uploads(files):
    imgs = []
    for f in files or []:
        path = f if isinstance(f, str) else f.name
        low = path.lower()
        if low.endswith(".npy"):
            a = np.load(path)
            if a.dtype != np.uint8:
                a = np.clip(a * (255 if a.max() <= 1.0 else 1), 0, 255).astype(np.uint8)
            if a.ndim == 4 and a.shape[1] == 3:
                a = a.transpose(0, 2, 3, 1)
            imgs.extend([np.asarray(Image.fromarray(x).convert("RGB").resize((32, 32), Image.BICUBIC)) for x in a])
        elif low.endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                for nm in sorted(z.namelist()):
                    if nm.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".webp")):
                        im = Image.open(io.BytesIO(z.read(nm))).convert("RGB").resize((32, 32), Image.BICUBIC)
                        imgs.append(np.asarray(im))
        else:
            imgs.append(np.asarray(Image.open(path).convert("RGB").resize((32, 32), Image.BICUBIC)))
    return np.stack(imgs) if imgs else None


def on_generate(choice, n, steps, seed):
    import torch
    from rdb.diffusion import ddim_sample, load_model
    if choice not in _MODELS:
        _MODELS.clear()
        _MODELS[choice] = load_model(ckpt_path(choice), DEVICE)[0]
    cfg = load_cfg(RUN)
    imgs = ddim_sample(_MODELS[choice], int(n), steps=int(steps), T=cfg["T"], schedule=cfg["schedule"],
                       batch=min(256, int(n)), seed=int(seed), device=DEVICE, progress=False)
    full = len(imgs) > cfg["knn_k"] + 1
    m, _, _ = measure(imgs, full=full)
    fig = composition_plot([("Original split", _ref().p_ref, REF_GRAY), (f"{choice} samples", m["p"], GEN_RAMP[2])],
                           f"Class composition of {len(imgs)} samples from {choice}")
    note = ("Small samples give noisy class shares; use n ≥ 500 for a stable β. "
            "The Results tab shows the full-size evaluation.")
    out = tempfile.NamedTemporaryFile(suffix=".npy", delete=False).name
    np.save(out, imgs)
    return grid_image(imgs[:100]), fig, metrics_table(m), note, out


def on_analyse(files, apply_kdr, keep_frac):
    imgs = load_uploads(files)
    if imgs is None:
        return None, None, [], "Upload images first.", None, None
    cfg = _STATE.get("cfg") or load_cfg(RUN)
    full = len(imgs) > cfg["knn_k"] + 1
    m, feats, probs = measure(imgs, full=full)
    series = [("Original split", _ref().p_ref, REF_GRAY), ("Uploaded set", m["p"], GEN_RAMP[1])]
    msg = f"{len(imgs)} images analysed."
    sel_grid, out = None, None
    if apply_kdr and full:
        from rdb.metrics import composition_metrics, class_hist
        from rdb.mitigation import kdr_weights, weighted_sample_without_replacement
        p = cfg["chains"]["kdr"]
        k = int(min(p.get("k", 50), max(2, len(imgs) // 4)))
        w, _ = kdr_weights(feats, _STATE["real_feats"], k=k, alpha=p.get("alpha", 1.0),
                           gamma=p.get("gamma", 1.0), clip_q=p.get("clip_q", 0.005), device=DEVICE)
        n_keep = max(1, int(round(keep_frac * len(imgs))))
        idx = weighted_sample_without_replacement(w, n_keep, np.random.default_rng(0))
        counts, _ = class_hist(probs[idx], cfg["conf_thresh"])
        ref = _ref()
        mc = composition_metrics(counts, ref.p_ref, ref.head_idx, ref.tail_idx)
        series.append((f"After KDR-IR (keep {n_keep})", mc["p"], CHAIN_COLORS["kdr"]))
        msg += (f"  KDR-IR resampling: β {m['beta']:.3f} → {mc['beta']:.3f},  "
                f"TRR {m['trr']:.3f} → {mc['trr']:.3f},  KL {m['kl']:.4f} → {mc['kl']:.4f}.")
        sel_grid = grid_image(imgs[np.sort(idx)][:100])
        out = tempfile.NamedTemporaryFile(suffix=".npy", delete=False).name
        np.save(out, imgs[np.sort(idx)])
    elif apply_kdr:
        msg += "  Need more images for KDR-IR."
    fig = composition_plot(series, "Class composition vs. the original imbalanced split")
    return grid_image(imgs[:100]), fig, metrics_table(m), msg, sel_grid, out


def results_view(choice):
    R = jload(os.path.join(RUN, "results", "metrics.json"))
    if choice == "gen0":
        m, tag = R["gen0"], "gen0"
    else:
        name, g = choice.split("/")
        m, tag = R["chains"][name][g[3:]], f"{name}_{g}"
    gd = os.path.join(RUN, "results", "grids")
    rnd = os.path.join(gd, f"random_{tag}.png")
    tail = os.path.join(gd, f"tail_gallery_{tag}.png")
    return (rnd if os.path.exists(rnd) else None, tail if os.path.exists(tail) else None, metrics_table(m))


def build_ui():
    import gradio as gr
    cfg = load_cfg(RUN)
    choices = model_choices()
    res_dir = os.path.join(RUN, "results")
    summary = open(os.path.join(res_dir, "summary.md")).read() if os.path.exists(os.path.join(res_dir, "summary.md")) \
        else "Run `python -m rdb.pipeline evaluate --run RUN` to produce results."
    figs = [os.path.join(res_dir, "figures", f) for f in sorted(os.listdir(os.path.join(res_dir, "figures")))] \
        if os.path.exists(os.path.join(res_dir, "figures")) else []
    split = jload(os.path.join(RUN, "data", "split.json"))
    split_md = "| class | group | images | share |\n|---|---|---|---|\n" + "\n".join(
        f"| {c['cls']} | {c['group']} | {c['count']} | {c['proportion']:.4f} |" for c in split["classes"])

    with gr.Blocks(title="Recursive density bias") as demo:
        gr.Markdown("# Higher-density bias under recursive diffusion training\n"
                    f"Run `{RUN}` · preset **{cfg['preset']}** · chains: {', '.join(cfg['run_chains'])} · "
                    f"features: `{cfg.get('feat_backbone_resolved')}`")
        with gr.Tab("Results"):
            gr.Markdown(summary)
            gr.Gallery(value=figs, label="Figures", columns=2, height="auto")
            with gr.Accordion("Controlled density-imbalanced split", open=False):
                gr.Markdown(split_md)
            sel = gr.Dropdown(choices, value=choices[0] if choices else None, label="Model")
            with gr.Row():
                g1 = gr.Image(label="100 random samples", type="filepath")
                g2 = gr.Image(label="Most confident tail-class samples (rows: " + ", ".join(cfg["tail"]) + ")",
                              type="filepath")
            tbl = gr.Dataframe(headers=["metric", "value"], label="Metrics (full-size evaluation)")
            sel.change(results_view, sel, [g1, g2, tbl])
            demo.load(results_view, sel, [g1, g2, tbl])
        with gr.Tab("Generate"):
            with gr.Row():
                mdl = gr.Dropdown(choices, value=choices[0] if choices else None, label="Model")
                n = gr.Slider(16, 1000, value=200, step=8, label="Number of images")
                st = gr.Slider(10, 200, value=50, step=5, label="DDIM steps")
                sd = gr.Number(value=0, label="Seed", precision=0)
            btn = gr.Button("Generate and measure", variant="primary")
            with gr.Row():
                img = gr.Image(label="Samples (first 100)")
                plot = gr.Plot(label="Class composition")
            tbl2 = gr.Dataframe(headers=["metric", "value"])
            note = gr.Markdown()
            dl = gr.File(label="Download samples (.npy, uint8 NHWC)")
            btn.click(on_generate, [mdl, n, st, sd], [img, plot, tbl2, note, dl])
        with gr.Tab("Analyse your images"):
            gr.Markdown("Upload generated images (PNG/JPG files, a .zip of images, or a uint8 `.npy` array "
                        "N×32×32×3). They are resized to 32×32 and compared with the original imbalanced split. "
                        "Tick KDR-IR to apply the proposed density-ratio resampling to your set.")
            up = gr.File(file_count="multiple", label="Images / .zip / .npy")
            with gr.Row():
                kd = gr.Checkbox(value=True, label="Apply KDR-IR resampling")
                kf = gr.Slider(0.2, 1.0, value=0.5, step=0.05, label="Fraction of images to keep")
            btn2 = gr.Button("Analyse", variant="primary")
            msg = gr.Markdown()
            with gr.Row():
                im_in = gr.Image(label="Uploaded (first 100)")
                im_sel = gr.Image(label="Kept by KDR-IR (first 100)")
            plot2 = gr.Plot()
            tbl3 = gr.Dataframe(headers=["metric", "value"])
            dl2 = gr.File(label="Download KDR-IR selection (.npy)")
            btn2.click(on_analyse, [up, kd, kf], [im_in, plot2, tbl3, msg, im_sel, dl2])
    return demo


def main():
    global RUN, DEVICE
    p = argparse.ArgumentParser()
    p.add_argument("--run", default=os.environ.get("RDB_RUN", "/kaggle/working/run"))
    p.add_argument("--share", action="store_true")
    p.add_argument("--port", type=int, default=7860)
    a = p.parse_args()
    RUN = os.path.abspath(a.run)
    DEVICE = get_device()
    build_ui().launch(share=a.share, server_port=a.port)


if __name__ == "__main__":
    main()

In [ ]:
import os, shutil, sys
os.chdir("/kaggle/working/recursive-density-bias")
if RESUME_FROM and not os.path.exists(os.path.join(RUN_DIR, "config.json")):
    print("copying previous run from", RESUME_FROM)
    shutil.copytree(RESUME_FROM, RUN_DIR, dirs_exist_ok=True)
extra = (f" --chains {CHAINS}" if CHAINS else "") + (f" --model {MODEL}" if MODEL else "")
cmd = f"{sys.executable} -m rdb.pipeline all --run {RUN_DIR} --preset {PRESET} --stages {STAGES}{extra}"
print(cmd)
!{cmd}

## Results

In [ ]:
from IPython.display import Markdown, Image, display
import glob, os
display(Markdown(open(f"{RUN_DIR}/results/summary.md").read()))
for f in sorted(glob.glob(f"{RUN_DIR}/results/figures/*.png")):
    print(os.path.basename(f)); display(Image(f))

In [ ]:
import json
cfg = json.load(open(f"{RUN_DIR}/config.json"))
G = cfg["gens"]
for tag in ["gen0"] + [f"{c}_gen{G}" for c in cfg["run_chains"]]:
    for kind in ["random", "tail_gallery"]:
        p = f"{RUN_DIR}/results/grids/{kind}_{tag}.png"
        if os.path.exists(p):
            print(kind, tag, "(tail rows: " + ", ".join(cfg["tail"]) + ")" if kind == "tail_gallery" else "")
            display(Image(p, width=520))

In [ ]:
import zipfile, glob, os
out = "/kaggle/working/deliverables.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("/kaggle/working/recursive-density-bias/**/*.py", recursive=True):
        z.write(f, os.path.relpath(f, "/kaggle/working"))
    for pat in ["config.json", "data/split*", "classifier/*.pt", "classifier/*.json", "classifier/*.npy",
                "gen0/ema.pt", "gen0/*.png", "gen0/*.csv", "chains/*/gen*/ema.pt", "chains/*/gen*/*.json",
                "chains/*/gen*/*.png", "chains/*/gen*/*.csv", "chains/*/gen*/train_idx.npy",
                "results/**/*", "logs/*"]:
        for f in glob.glob(f"{RUN_DIR}/{pat}", recursive=True):
            if os.path.isfile(f):
                z.write(f, os.path.join("run", os.path.relpath(f, RUN_DIR)))
print(out, f"{os.path.getsize(out) / 1e6:.0f} MB")

In [ ]:
import os, subprocess, sys
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Batch":
    print("Committed run: skipping the interface. Open the notebook interactively to launch it.")
else:
    os.chdir("/kaggle/working/recursive-density-bias")
    !pip install -q gradio 2>&1 | tail -1
    !{sys.executable} app.py --run {RUN_DIR} --share